# **สร้างโครงสร้างโปรเจกต์**

สร้างโฟลเดอร์สำหรับแยกไฟล์ Python และเก็บข้อมูล JSON

In [1]:
from pathlib import Path
import json
import sys
import datetime

PROJECT_DIR = Path("/content/final-task-manager-project")
SRC_DIR = PROJECT_DIR / "src"
DATA_DIR = PROJECT_DIR / "data"

SRC_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder created:", PROJECT_DIR)

Project folder created: /content/final-task-manager-project


# **สร้าง task.py**

สร้างคลาส Task, DueDateTask และ PriorityTask พร้อมระบบ Tags

In [2]:
(SRC_DIR / "task.py").write_text('''
import datetime


class Task:
    def __init__(self, id, description, completed=False, tags=None):
        self.id = int(id)
        self.description = str(description).strip()

        if not self.description:
            raise ValueError("Description cannot be empty.")

        self.completed = bool(completed)
        self.tags = []

        for tag in (tags or []):
            self.add_tag(tag)

    def mark_complete(self):
        self.completed = True

    def add_tag(self, tag):
        tag = str(tag).strip().lower()
        if tag and tag not in self.tags:
            self.tags.append(tag)

    def remove_tag(self, tag):
        tag = str(tag).strip().lower()
        if tag in self.tags:
            self.tags.remove(tag)

    def to_dict(self):
        return {
            "id": self.id,
            "description": self.description,
            "completed": self.completed,
            "tags": self.tags,
            "_type": "Task"
        }

    def __str__(self):
        status = "Completed" if self.completed else "Pending"
        return (
            f"ID: {self.id} | {self.description} | "
            f"{status} | Tags: {', '.join(self.tags)}"
        )


class DueDateTask(Task):
    def __init__(self, id, description, due_date,
                 completed=False, tags=None):
        super().__init__(id, description, completed, tags)

        if isinstance(due_date, str):
            due_date = datetime.date.fromisoformat(due_date)

        self.due_date = due_date

    def to_dict(self):
        data = super().to_dict()
        data["due_date"] = self.due_date.isoformat()
        data["_type"] = "DueDateTask"
        return data

    def __str__(self):
        return f"{super().__str__()} | Due: {self.due_date}"


class PriorityTask(Task):
    ALLOWED_PRIORITIES = ("High", "Medium", "Low")

    def __init__(self, id, description, priority,
                 completed=False, tags=None):
        super().__init__(id, description, completed, tags)

        priority = str(priority).strip().capitalize()

        if priority not in self.ALLOWED_PRIORITIES:
            raise ValueError("Priority must be High, Medium, or Low.")

        self.priority = priority

    def to_dict(self):
        data = super().to_dict()
        data["priority"] = self.priority
        data["_type"] = "PriorityTask"
        return data

    def __str__(self):
        return f"{super().__str__()} | Priority: {self.priority}"
''', encoding="utf-8")

print("task.py created")

task.py created


# **สร้าง data_persistence.py**

จัดการบันทึกและโหลดข้อมูลจาก JSON โดยสร้าง Object กลับมาให้ตรงกับประเภทงาน

In [3]:
(SRC_DIR / "data_persistence.py").write_text('''
import json
from pathlib import Path
from .task import Task, DueDateTask, PriorityTask


class DataPersistence:
    def __init__(self, data_file="data/tasks.json"):
        root = Path(__file__).resolve().parent.parent
        self.data_file = Path(data_file)

        if not self.data_file.is_absolute():
            self.data_file = root / self.data_file

    def load_tasks(self):
        if not self.data_file.exists():
            return []

        try:
            with self.data_file.open("r", encoding="utf-8") as file:
                records = json.load(file)

            tasks = []

            for record in records:
                record = dict(record)
                task_type = record.pop("_type", "Task")
                record.setdefault("tags", [])

                if task_type == "Task":
                    tasks.append(Task(**record))
                elif task_type == "DueDateTask":
                    tasks.append(DueDateTask(**record))
                elif task_type == "PriorityTask":
                    tasks.append(PriorityTask(**record))

            return tasks

        except (json.JSONDecodeError, OSError, ValueError, TypeError) as error:
            print("Could not load tasks:", error)
            return []

    def save_tasks(self, tasks):
        self.data_file.parent.mkdir(parents=True, exist_ok=True)

        with self.data_file.open("w", encoding="utf-8") as file:
            json.dump(
                [task.to_dict() for task in tasks],
                file,
                indent=2,
                ensure_ascii=False
            )
''', encoding="utf-8")

print("data_persistence.py created")

data_persistence.py created


# **สร้าง task_manager.py**

ควบคุมการเพิ่ม ลบ ทำเครื่องหมายเสร็จ ค้นหา กรอง และเรียงลำดับงาน

In [4]:
(SRC_DIR / "task_manager.py").write_text('''
import datetime
from .task import Task, DueDateTask, PriorityTask
from .data_persistence import DataPersistence


class TaskManager:
    PRIORITY_ORDER = {"High": 1, "Medium": 2, "Low": 3}

    def __init__(self, data_file="data/tasks.json"):
        self.persistence = DataPersistence(data_file)
        self.tasks = self.persistence.load_tasks()
        self.next_id = max((t.id for t in self.tasks), default=0) + 1

    def save(self):
        self.persistence.save_tasks(self.tasks)

    def add_normal_task(self, description, tags=None):
        task = Task(self.next_id, description, tags=tags)
        self.tasks.append(task)
        self.next_id += 1
        self.save()
        return task

    def add_due_date_task(self, description, due_date, tags=None):
        task = DueDateTask(
            self.next_id, description, due_date, tags=tags
        )
        self.tasks.append(task)
        self.next_id += 1
        self.save()
        return task

    def add_priority_task(self, description, priority, tags=None):
        task = PriorityTask(
            self.next_id, description, priority, tags=tags
        )
        self.tasks.append(task)
        self.next_id += 1
        self.save()
        return task

    def get_task_by_id(self, task_id):
        try:
            task_id = int(task_id)
        except (TypeError, ValueError):
            return None

        return next(
            (t for t in self.tasks if t.id == task_id),
            None
        )

    def get_all_tasks(self):
        return list(self.tasks)

    def complete_task(self, task_id):
        task = self.get_task_by_id(task_id)

        if task is None:
            return False, "Task not found."

        task.mark_complete()
        self.save()
        return True, "Task completed."

    def delete_task(self, task_id):
        task = self.get_task_by_id(task_id)

        if task is None:
            return False, "Task not found."

        self.tasks.remove(task)
        self.save()
        return True, "Task deleted."

    def search_tasks(self, keyword):
        keyword = str(keyword).strip().casefold()

        if not keyword:
            return []

        return [
            task for task in self.tasks
            if keyword in task.description.casefold()
            or any(keyword in tag.casefold() for tag in task.tags)
        ]

    def filter_tasks(
        self,
        status=None,
        due_date_before=None,
        due_date_after=None,
        priority_level=None,
        contains_tags=None
    ):
        result = list(self.tasks)

        if status is not None:
            result = [t for t in result if t.completed == status]

        if due_date_before:
            before = datetime.date.fromisoformat(due_date_before)
            result = [
                t for t in result
                if isinstance(t, DueDateTask) and t.due_date <= before
            ]

        if due_date_after:
            after = datetime.date.fromisoformat(due_date_after)
            result = [
                t for t in result
                if isinstance(t, DueDateTask) and t.due_date >= after
            ]

        if priority_level:
            priority_level = priority_level.capitalize()
            result = [
                t for t in result
                if isinstance(t, PriorityTask)
                and t.priority == priority_level
            ]

        if contains_tags:
            required = [
                tag.strip().lower()
                for tag in contains_tags if tag.strip()
            ]

            result = [
                t for t in result
                if all(tag in t.tags for tag in required)
            ]

        return result

    def sort_tasks(self, criterion="id", reverse=False):
        tasks = list(self.tasks)

        if criterion == "due_date":
            dated = [t for t in tasks if isinstance(t, DueDateTask)]
            other = [t for t in tasks if not isinstance(t, DueDateTask)]
            dated.sort(key=lambda t: t.due_date, reverse=reverse)
            return dated + other

        if criterion == "priority":
            prioritized = [
                t for t in tasks if isinstance(t, PriorityTask)
            ]
            other = [t for t in tasks if not isinstance(t, PriorityTask)]

            prioritized.sort(
                key=lambda t: self.PRIORITY_ORDER[t.priority],
                reverse=reverse
            )
            return prioritized + other

        keys = {
            "id": lambda t: t.id,
            "description": lambda t: t.description.casefold(),
            "completed": lambda t: t.completed
        }

        return sorted(
            tasks,
            key=keys.get(criterion, keys["id"]),
            reverse=reverse
        )
''', encoding="utf-8")

print("task_manager.py created")

task_manager.py created


# **สร้าง cli_interface.py**

สร้างเมนูรับคำสั่งจากผู้ใช้ผ่าน Command Line

In [5]:
(SRC_DIR / "cli_interface.py").write_text('''
import datetime


class CLIInterface:
    def __init__(self, task_manager):
        self.manager = task_manager

    def display_menu(self):
        print("\\\\n=== Enhanced Task Manager ===")
        print("1. Add Normal Task")
        print("2. Add Due Date Task")
        print("3. Add Priority Task")
        print("4. List Tasks")
        print("5. Complete Task")
        print("6. Delete Task")
        print("7. Search Tasks")
        print("8. Filter Tasks")
        print("9. Sort Tasks")
        print("0. Exit")

    def get_description(self):
        while True:
            description = input("Description: ").strip()

            if description:
                return description

            print("Description cannot be empty.")

    def get_tags(self):
        raw = input("Tags separated by commas (optional): ")
        return [
            tag.strip().lower()
            for tag in raw.split(",") if tag.strip()
        ]

    def display_tasks(self, tasks):
        if not tasks:
            print("No tasks found.")
            return

        for task in tasks:
            print(task)

    def run(self):
        while True:
            self.display_menu()
            choice = input("Select menu: ").strip()

            try:
                if choice == "1":
                    task = self.manager.add_normal_task(
                        self.get_description(), self.get_tags()
                    )
                    print("Added:", task)

                elif choice == "2":
                    description = self.get_description()
                    due_date = input("Due date (YYYY-MM-DD): ").strip()
                    datetime.date.fromisoformat(due_date)
                    task = self.manager.add_due_date_task(
                        description, due_date, self.get_tags()
                    )
                    print("Added:", task)

                elif choice == "3":
                    description = self.get_description()
                    priority = input("Priority (High/Medium/Low): ")
                    task = self.manager.add_priority_task(
                        description, priority, self.get_tags()
                    )
                    print("Added:", task)

                elif choice == "4":
                    self.display_tasks(self.manager.get_all_tasks())

                elif choice == "5":
                    task_id = int(input("Task ID: "))
                    print(self.manager.complete_task(task_id)[1])

                elif choice == "6":
                    task_id = int(input("Task ID: "))
                    print(self.manager.delete_task(task_id)[1])

                elif choice == "7":
                    keyword = input("Search keyword: ")
                    self.display_tasks(
                        self.manager.search_tasks(keyword)
                    )

                elif choice == "8":
                    status_text = input(
                        "Status (completed/pending/blank): "
                    ).strip().lower()

                    status = (
                        True if status_text == "completed"
                        else False if status_text == "pending"
                        else None
                    )

                    priority = input(
                        "Priority (High/Medium/Low/blank): "
                    ).strip() or None

                    tags_text = input(
                        "Required tags, comma-separated (blank to skip): "
                    )
                    tags = [
                        t.strip() for t in tags_text.split(",") if t.strip()
                    ] or None

                    tasks = self.manager.filter_tasks(
                        status=status,
                        priority_level=priority,
                        contains_tags=tags
                    )
                    self.display_tasks(tasks)

                elif choice == "9":
                    criterion = input(
                        "Sort by id/description/completed/due_date/priority: "
                    ).strip().lower() or "id"

                    reverse = (
                        input("Descending? (yes/no): ").strip().lower()
                        == "yes"
                    )

                    self.display_tasks(
                        self.manager.sort_tasks(criterion, reverse)
                    )

                elif choice == "0":
                    print("Goodbye!")
                    break

                else:
                    print("Invalid menu choice.")

            except (ValueError, TypeError) as error:
                print("Invalid input:", error)
''', encoding="utf-8")

print("cli_interface.py created")

cli_interface.py created


# **สร้าง main.py**

เป็นจุดเริ่มต้นของโปรแกรม และสร้างไฟล์ __init__.py สำหรับแพ็กเกจ

In [6]:
(SRC_DIR / "__init__.py").write_text(
    '"""Enhanced Task Manager package."""\n',
    encoding="utf-8"
)

(PROJECT_DIR / "main.py").write_text('''
from src.task_manager import TaskManager
from src.cli_interface import CLIInterface


def main():
    manager = TaskManager("data/tasks.json")
    interface = CLIInterface(manager)
    interface.run()


if __name__ == "__main__":
    main()
''', encoding="utf-8")

print("main.py created")

main.py created


# **สร้างข้อมูลตัวอย่าง tasks.json**

ใช้ทดสอบงานทั่วไป งานที่มีวันกำหนดส่ง และงานที่มีระดับความสำคัญ

In [7]:
sample_tasks = [
    {
        "id": 1,
        "description": "Finish Python assignment",
        "completed": False,
        "tags": ["school", "urgent"],
        "_type": "Task"
    },
    {
        "id": 2,
        "description": "Submit project report",
        "completed": False,
        "tags": ["school", "project"],
        "due_date": "2026-10-20",
        "_type": "DueDateTask"
    },
    {
        "id": 3,
        "description": "Fix critical bug",
        "completed": True,
        "tags": ["coding", "urgent"],
        "priority": "High",
        "_type": "PriorityTask"
    },
    {
        "id": 4,
        "description": "Clean workspace",
        "completed": False,
        "tags": ["home"],
        "priority": "Low",
        "_type": "PriorityTask"
    }
]

(DATA_DIR / "tasks.json").write_text(
    json.dumps(sample_tasks, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Sample tasks created")

Sample tasks created


# **ทดสอบฟังก์ชันหลัก**

In [8]:
sys.path.insert(0, str(PROJECT_DIR))

from src.task_manager import TaskManager

manager = TaskManager("data/tasks.json")

print("=== All Tasks ===")
for task in manager.get_all_tasks():
    print(task)

print("\\n=== Search: urgent ===")
for task in manager.search_tasks("urgent"):
    print(task)

print("\\n=== Filter: High Priority ===")
for task in manager.filter_tasks(priority_level="High"):
    print(task)

print("\\n=== Filter: school tag ===")
for task in manager.filter_tasks(contains_tags=["school"]):
    print(task)

print("\\n=== Sort by Due Date ===")
for task in manager.sort_tasks("due_date"):
    print(task)

=== All Tasks ===
ID: 1 | Finish Python assignment | Pending | Tags: school, urgent
ID: 2 | Submit project report | Pending | Tags: school, project | Due: 2026-10-20
ID: 3 | Fix critical bug | Completed | Tags: coding, urgent | Priority: High
ID: 4 | Clean workspace | Pending | Tags: home | Priority: Low
\n=== Search: urgent ===
ID: 1 | Finish Python assignment | Pending | Tags: school, urgent
ID: 3 | Fix critical bug | Completed | Tags: coding, urgent | Priority: High
\n=== Filter: High Priority ===
ID: 3 | Fix critical bug | Completed | Tags: coding, urgent | Priority: High
\n=== Filter: school tag ===
ID: 1 | Finish Python assignment | Pending | Tags: school, urgent
ID: 2 | Submit project report | Pending | Tags: school, project | Due: 2026-10-20
\n=== Sort by Due Date ===
ID: 2 | Submit project report | Pending | Tags: school, project | Due: 2026-10-20
ID: 1 | Finish Python assignment | Pending | Tags: school, urgent
ID: 3 | Fix critical bug | Completed | Tags: coding, urgent | Pri

# **ทดสอบการบันทึกข้อมูลและตรวจสอบผล**

In [9]:
# Mark task 1 as completed
print(manager.complete_task(1))

# Add a new priority task
new_task = manager.add_priority_task(
    "Prepare final presentation",
    "High",
    tags=["school", "presentation"]
)
print("Added:", new_task)

# Reload from JSON to verify persistence
manager_reloaded = TaskManager("data/tasks.json")

print("\\n=== Reloaded Tasks ===")
for task in manager_reloaded.get_all_tasks():
    print(task)

assert manager_reloaded.get_task_by_id(1).completed is True
assert manager_reloaded.get_task_by_id(new_task.id) is not None

print("\\nPersistence checks passed.")

(True, 'Task completed.')
Added: ID: 5 | Prepare final presentation | Pending | Tags: school, presentation | Priority: High
\n=== Reloaded Tasks ===
ID: 1 | Finish Python assignment | Completed | Tags: school, urgent
ID: 2 | Submit project report | Pending | Tags: school, project | Due: 2026-10-20
ID: 3 | Fix critical bug | Completed | Tags: coding, urgent | Priority: High
ID: 4 | Clean workspace | Pending | Tags: home | Priority: Low
ID: 5 | Prepare final presentation | Pending | Tags: school, presentation | Priority: High
\nPersistence checks passed.


# **Automated checks**

In [10]:
check_manager = TaskManager("data/tasks.json")

assert check_manager.search_tasks("URGENT")
assert check_manager.filter_tasks(priority_level="High")
assert check_manager.filter_tasks(contains_tags=["school"])

ids = [task.id for task in check_manager.sort_tasks("id")]
assert ids == sorted(ids)

try:
    check_manager.add_priority_task(
        "Invalid priority test",
        "Urgent"
    )
    raise AssertionError("Expected invalid priority to fail")
except ValueError:
    pass

print("All automated checks passed.")

All automated checks passed.


# **สร้าง README และ PLAN**

In [12]:

from pathlib import Path

PROJECT_DIR = Path("/content/final-task-manager-project")

readme = """# Enhanced Task Manager CLI

A modular command-line task management application built with Python.

## Features
- Create, view, update, and delete tasks
- Set task status and due dates
- Assign priority levels
- Add tags to tasks
- Search, filter, and sort tasks
- Save and load tasks using JSON
- Support different task types

## Project Structure

```text
final-task-manager-project/
├── main.py
├── README.md
├── PLAN.md
├── .gitignore
├── src/
│   ├── __init__.py
│   ├── task.py
│   ├── task_manager.py
│   ├── data_persistence.py
│   └── cli_interface.py
└── data/
    └── tasks.json
```

## Requirements
- Python 3.10 or later

## How to Run

Open a terminal in the project directory and run:

```bash
python main.py
```

## Data Storage
Tasks are stored in `data/tasks.json`.
"""

plan = """# Project Plan

## Objective
Develop a modular command-line task manager using Python.

## Main Components
1. Task models
2. Task management
3. JSON data persistence
4. Command-line interface
5. Testing and documentation

## Development Steps
- [x] Create project structure
- [x] Implement task models
- [x] Implement task management functions
- [x] Implement JSON persistence
- [x] Implement command-line interface
- [x] Test core functionality
- [x] Create documentation
"""

gitignore = """__pycache__/
*.py[cod]
.venv/
.env
"""

(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
(PROJECT_DIR / "PLAN.md").write_text(plan, encoding="utf-8")
(PROJECT_DIR / ".gitignore").write_text(gitignore, encoding="utf-8")

print("Created README.md, PLAN.md, and .gitignore")


Created README.md, PLAN.md, and .gitignore


# **ตรวจสอบไฟล์ทั้งหมด**

In [16]:
from pathlib import Path

PROJECT_DIR = Path("/content/final-task-manager-project")

print("Project files:")
for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_file():
        print(f"- {path.relative_to(PROJECT_DIR)}")

print("\nREADME exists:", (PROJECT_DIR / "README.md").exists())
print("PLAN exists:", (PROJECT_DIR / "PLAN.md").exists())
print("Main file exists:", (PROJECT_DIR / "main.py").exists())


Project files:
- .gitignore
- PLAN.md
- README.md
- data/tasks.json
- main.py
- src/__init__.py
- src/__pycache__/__init__.cpython-313.pyc
- src/__pycache__/data_persistence.cpython-313.pyc
- src/__pycache__/task.cpython-313.pyc
- src/__pycache__/task_manager.cpython-313.pyc
- src/cli_interface.py
- src/data_persistence.py
- src/task.py
- src/task_manager.py

README exists: True
PLAN exists: True
Main file exists: True
